# Face Recognition using Projections, Eigenvectors and PCA (Eigenfaces)
### UE25MA242A – Mathematical Foundation for AI & Data Science · Mini Project
**Problem Statement 11:** *Projections, eigenvectors, Principal Component Analysis, and face recognition algorithms*

**In one sentence:** we turn photos of faces into a matrix, use the eigenvectors of its covariance matrix (PCA) to find the
main face patterns ("eigenfaces"), project every face onto them, and recognise **who** is in a new photo.

**Dataset:** Olivetti faces: 400 photos of 40 people (10 each), 64 × 64 pixels.
We use 8 photos per person for training (320) and keep 2 per person for testing (80).

| Step | Linear algebra used | Output |
|---|---|---|
| 1 | Matrix representation | 320 × 4096 data matrix |
| 2 | Mean vector, centering | The average face |
| 3 | Rank, linear independence | How much of the data is redundant |
| 4 | Covariance, eigenvalues, eigenvectors, diagonalization (PCA) | The main face patterns |
| 5 | Eigenfaces | The patterns shown as pictures |
| 6 | **Projection** onto a subspace | Faces compressed from 4096 to 50 numbers |
| 7 | Distance (norm) between projections | **Face recognition** |
| 8 | Least squares | A second recognition method |
| 9 | CNN | A deep learning comparison |

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_olivetti_faces
import torch
import torch.nn as nn

np.random.seed(0)
torch.manual_seed(0)
np.set_printoptions(precision=2, suppress=True, linewidth=140)

## Step 1 · Faces as a Matrix

**Concept:** A 64 × 64 grayscale photo is a matrix of brightness values (0 = black, 1 = white).
We flatten each photo into one row of 64 × 64 = 4096 numbers and stack the rows into a **data matrix**.

**Purpose:** Once faces are vectors, we can apply linear algebra to them.

**Outcome:** A 320 × 4096 training matrix $X$ (one row per photo) and an 80 × 4096 test matrix.

In [ ]:
faces = fetch_olivetti_faces(shuffle=False)
images = faces.images.astype(float)             # shape (400, 64, 64), values in [0, 1]
people = faces.target                           # person number 0-39 for each photo

is_train = np.arange(400) % 10 < 8              # first 8 photos of each person -> training
train_imgs, y_train = images[is_train],  people[is_train]
test_imgs,  y_test  = images[~is_train], people[~is_train]

X      = train_imgs.reshape(-1, 4096)           # training data matrix
X_test = test_imgs.reshape(-1, 4096)
n = X.shape[0]

print("Training matrix X:", X.shape, " (rows = photos, columns = pixels)")
print("Test matrix      :", X_test.shape)
print("Number of people :", len(np.unique(people)))

In [ ]:
face = train_imgs[0]
print("A 6x6 block of pixel values from the first face (around the left eye):\n")
print(face[22:28, 16:22])

fig, ax = plt.subplots(1, 8, figsize=(14, 2.3))
for p, a in enumerate(ax):
    a.imshow(train_imgs[p * 8], cmap="gray"); a.set_title(f"person {p}"); a.axis("off")
plt.suptitle("One training photo of each of the first 8 people"); plt.show()

> **Say this in viva:** *"Each photo is a matrix of pixel values. We flattened each one into a vector of 4096 numbers,
> so our training set became a 320 × 4096 matrix, with one row per photo."*

## Step 2 · The Average Face and Centering

**Concept:** The **mean vector** $m$ is the average of all rows of $X$. Subtracting it from every row (**centering**)
gives $X_c = X - m$, where each row shows how one face *differs* from the average face.

**Purpose:** PCA looks for the ways faces differ from each other, so we must remove what they all have in common first.

**Outcome:** The average face, and centered faces that show only each person's unique features.

In [ ]:
mean_face = X.mean(axis=0)                      # average of all 320 rows
Xc = X - mean_face                              # centered data

fig, ax = plt.subplots(1, 3, figsize=(8, 3))
ax[0].imshow(mean_face.reshape(64, 64), cmap="gray");  ax[0].set_title("Average face  m")
ax[1].imshow(face, cmap="gray");                       ax[1].set_title("One face  x")
ax[2].imshow(Xc[0].reshape(64, 64), cmap="bwr");       ax[2].set_title("x − m  (difference)")
for a in ax: a.axis("off")
plt.tight_layout(); plt.show()

> **Say this in viva:** *"We averaged all the faces to get the average face, then subtracted it from each face.
> What's left is only what makes each face different, which is what we need to recognise people."*

## Step 3 · Rank and Redundancy

**Concept:** The **rank** of a matrix is the number of linearly independent rows (or columns). It can never be more
than the number of rows or the number of columns: $\text{rank}(X) \le \min(320, 4096)$.

**Purpose:** Find out how many directions the face data really uses.

**Outcome:** Although each face has 4096 numbers, the data spans at most 320 independent directions,
so at least 3776 of the 4096 directions carry no information from our data.

In [ ]:
rank = np.linalg.matrix_rank(X)
print(f"Number of columns (pixels):          4096")
print(f"Number of rows (photos):             {n}")
print(f"Rank of X:                           {rank}")
print(f"Unused directions (4096 − rank):     {4096 - rank}")

> **Say this in viva:** *"Each face lives in a 4096-dimensional space, but we only have 320 faces, so the rank is at most 320.
> The faces only fill a small subspace, and that's why we can describe them with far fewer numbers."*

## Step 4 · Covariance Matrix, Eigenvectors and Diagonalization (PCA)

**Concept:**
1. The **covariance matrix** $C = \frac{1}{n-1} X_c^T X_c$ (4096 × 4096) records how pixels vary together.
2. $C$ is **symmetric**, so it can be **diagonalized**: $C = P D P^T$. The columns of $P$ are orthonormal
   **eigenvectors** ($Cv = \lambda v$) and $D$ holds the **eigenvalues**.
3. Each eigenvector is a direction in which faces vary; its eigenvalue says how much they vary in that direction.

**Purpose:** Find the most important directions of variation. This is **Principal Component Analysis**.

**Outcome:** 4096 eigenvectors sorted by importance, with checks that the diagonalization is correct.

In [ ]:
C = Xc.T @ Xc / (n - 1)                         # covariance matrix, 4096 x 4096

eigvals, P = np.linalg.eigh(C)                  # eigh: for symmetric matrices (takes ~30 s)
order = np.argsort(eigvals)[::-1]               # largest eigenvalue first
eigvals, P = eigvals[order], P[:, order]

print("C is symmetric:            ", np.allclose(C, C.T))
print("Diagonalization C = P D Pᵀ:", np.allclose(C, P @ np.diag(eigvals) @ P.T))
print("Eigenvectors orthonormal:  ", np.allclose(P.T @ P, np.eye(4096)))

nonzero = np.sum(eigvals > 1e-10 * eigvals[0])
print(f"\nNon-zero eigenvalues: {nonzero}   (rank of the centered data = {np.linalg.matrix_rank(Xc)})")
print("Top 5 eigenvalues:", eigvals[:5])

> **Say this in viva:** *"The covariance matrix is symmetric, so we diagonalized it into orthogonal eigenvectors.
> Each eigenvector is a direction in which faces vary, and its eigenvalue says how much. Only 319 eigenvalues are non-zero,
> matching the rank of the centered data (one direction is lost when we subtract the mean), so the rank from Step 3 shows up again here."*

## Step 5 · Eigenfaces

**Concept:** Each eigenvector has 4096 numbers, so it can be reshaped into a 64 × 64 picture: an **eigenface**.
The share of variation explained by eigenvector $i$ is $\lambda_i / \sum \lambda$.

**Purpose:** See the main face patterns and decide how many of them to keep.

**Outcome:** The top eigenfaces, and the number needed to explain 90% of the variation.

In [ ]:
explained = eigvals / eigvals.sum()
cumulative = np.cumsum(explained)
k90 = np.argmax(cumulative >= 0.90) + 1
print(f"Only {k90} of 4096 eigenvectors are needed to explain 90% of the variation.")

fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(np.arange(1, 321), cumulative[:320]); ax.axhline(0.9, ls="--", c="r"); ax.axvline(k90, ls="--", c="r")
ax.set_xlabel("number of eigenvectors kept"); ax.set_ylabel("variation explained")
ax.set_title("How many eigenfaces do we need?"); plt.show()

fig, ax = plt.subplots(1, 8, figsize=(14, 2.3))
for i, a in enumerate(ax):
    a.imshow(P[:, i].reshape(64, 64), cmap="bwr"); a.set_title(f"eigenface {i+1}"); a.axis("off")
plt.suptitle("Top 8 eigenvectors shown as pictures"); plt.show()

> **Say this in viva:** *"Each eigenvector reshaped into a picture is an eigenface: a basic face pattern like lighting direction,
> face shape or glasses. Every face is roughly the average face plus some amount of each eigenface."*

## Step 6 · Projection onto the Eigenface Subspace

**Concept:** The top $k$ eigenvectors $P_k$ span a $k$-dimensional **subspace**. Because they are orthonormal, the
**orthogonal projection** of a face onto this subspace is found with dot products alone:

$$z = P_k^T (x - m) \qquad\qquad \hat{x} = m + P_k\, z$$

$z$ holds $k$ numbers (the face's coordinates in the subspace), and $\hat{x}$ is the face rebuilt from them.

**Purpose:** Describe every face with only $k$ numbers instead of 4096.

**Outcome:** Test faces (never seen during training) rebuilt from 5, 20, 50 and 150 numbers.

In [ ]:
def project(x, k):   return P[:, :k].T @ (x - mean_face)      # 4096 numbers -> k numbers
def rebuild(z, k):   return mean_face + P[:, :k] @ z          # k numbers -> 4096 numbers

ks = [5, 20, 50, 150]
fig, ax = plt.subplots(3, len(ks) + 1, figsize=(11, 7))
for r in range(3):
    x = X_test[r * 2]
    ax[r, 0].imshow(x.reshape(64, 64), cmap="gray"); ax[r, 0].set_title("Original\n4096 numbers" if r == 0 else "")
    for c, k in enumerate(ks, start=1):
        ax[r, c].imshow(rebuild(project(x, k), k).reshape(64, 64), cmap="gray")
        if r == 0: ax[r, c].set_title(f"k = {k}\n({4096/k:.0f}x smaller)")
for a in ax.flat: a.axis("off")
plt.tight_layout(); plt.show()

> **Say this in viva:** *"We projected each face onto the subspace spanned by the top eigenfaces. Each number in the projection
> says how much of one eigenface the face contains. With 50 numbers instead of 4096, new faces are still clearly recognisable."*

## Step 7 · Face Recognition with Eigenfaces

**Concept:** This is the **eigenfaces face recognition algorithm**. Project every training face and the new test face onto
the top $K = 50$ eigenfaces. Then find the training face whose projection is **closest**, measured by the distance (norm):

$$\text{distance} = \| z_{\text{test}} - z_{\text{train}} \|$$

The test face is recognised as the person in the closest training photo.

**Purpose:** Recognise who is in a photo the model has never seen.

**Outcome:** The recognition accuracy on the 80 test photos, and examples of matches.

In [ ]:
K = 50
Z_train = Xc @ P[:, :K]                          # projections of all 320 training faces (320 x 50)
Z_test  = (X_test - mean_face) @ P[:, :K]        # projections of all 80 test faces (80 x 50)

pred_ef, match = [], []
for z in Z_test:
    distances = np.linalg.norm(Z_train - z, axis=1)   # distance to every training face
    j = np.argmin(distances)                          # closest one
    match.append(j); pred_ef.append(y_train[j])
pred_ef = np.array(pred_ef)
acc_ef = np.mean(pred_ef == y_test)
print(f"Eigenfaces recognition accuracy ({K} eigenfaces): {acc_ef:.1%}  ({np.sum(pred_ef == y_test)} of {len(y_test)} correct)")

fig, ax = plt.subplots(2, 8, figsize=(14, 4))
for c, t in enumerate(range(0, 80, 10)):
    ok = pred_ef[t] == y_test[t]
    ax[0, c].imshow(test_imgs[t], cmap="gray"); ax[0, c].set_title(f"test: person {y_test[t]}", fontsize=9)
    ax[1, c].imshow(train_imgs[match[t]], cmap="gray")
    ax[1, c].set_title(f"match: person {pred_ef[t]}", fontsize=9, color="green" if ok else "red")
for a in ax.flat: a.axis("off")
plt.suptitle("Top: new test photo.  Bottom: closest training photo in eigenface space"); plt.tight_layout(); plt.show()

> **Say this in viva:** *"To recognise a face, we project it onto the 50 eigenfaces and find the training face whose projection
> is closest. Comparing 50 numbers instead of 4096 is faster, and it ignores small details that don't help identify a person."*

## Step 8 · Recognition with Least Squares

**Concept:** **Least squares** finds weights $W$ that make $AW$ as close as possible to $Y$: $\min_W \|AW - Y\|^2$.
$A$ holds each training face's 50 projection numbers (plus a 1 for a constant term), and $Y$ has one column per person,
with a 1 in the correct person's column. The solution satisfies the normal equations $A^TAW = A^TY$, and $AW$ is the
**projection** of $Y$ onto the column space of $A$.

**Purpose:** A second way to recognise faces from their projections.

**Outcome:** A test accuracy to compare with Step 7.

In [ ]:
A      = np.hstack([Z_train, np.ones((n, 1))])              # 320 x 51
A_test = np.hstack([Z_test,  np.ones((len(y_test), 1))])
Y = np.eye(40)[y_train]                                     # 320 x 40, one-hot

W, *_ = np.linalg.lstsq(A, Y, rcond=None)                   # least squares solution
pred_ls = np.argmax(A_test @ W, axis=1)                     # highest score = predicted person
acc_ls = np.mean(pred_ls == y_test)
print(f"Least squares recognition accuracy: {acc_ls:.1%}")

> **Say this in viva:** *"Least squares finds the best-fit linear model from the 50 projection numbers to the person's identity.
> It gives each photo 40 scores, one per person, and we pick the highest."*

## Step 9 · Comparison with a CNN

A small Convolutional Neural Network learns directly from the raw 64 × 64 photos. It has two layers of 3 × 3 filters,
each followed by ReLU and max pooling, and a final layer that gives a score for each of the 40 people.

In [ ]:
Xt = torch.tensor(train_imgs, dtype=torch.float32).unsqueeze(1)   # (320, 1, 64, 64)
yt = torch.tensor(y_train)
Xv = torch.tensor(test_imgs,  dtype=torch.float32).unsqueeze(1)

cnn = nn.Sequential(
    nn.Conv2d(1, 8, 3), nn.ReLU(), nn.MaxPool2d(2),       # 64 -> 62 -> 31
    nn.Conv2d(8, 16, 3), nn.ReLU(), nn.MaxPool2d(2),      # 31 -> 29 -> 14
    nn.Flatten(), nn.Linear(16 * 14 * 14, 40),
)
opt = torch.optim.Adam(cnn.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()

for epoch in range(30):
    perm = torch.randperm(n)
    for i in range(0, n, 32):
        idx = perm[i:i + 32]
        opt.zero_grad()
        loss = loss_fn(cnn(Xt[idx]), yt[idx])
        loss.backward(); opt.step()
    if (epoch + 1) % 10 == 0:
        print(f"epoch {epoch + 1}: loss = {loss.item():.3f}")

with torch.no_grad():
    acc_cnn = (cnn(Xv).argmax(1).numpy() == y_test).mean()
print(f"\nCNN recognition accuracy: {acc_cnn:.1%}")

> **Say this in viva:** *"The CNN learns its own filters from raw pixels, while eigenfaces uses a fixed mathematical recipe.
> With only 8 photos per person, the CNN has very little data to learn from, whereas eigenfaces works well even with small datasets."*

## Final Results

In [ ]:
print("=" * 60)
print(f"{'Training matrix X':<40}{str(X.shape):>20}")
print(f"{'Rank of X (out of 4096 columns)':<40}{rank:>20}")
print(f"{'Eigenfaces for 90% of variation':<40}{k90:>20}")
print(f"{'Projection used for recognition':<40}{f'4096 -> {K} ({4096/K:.0f}x)':>20}")
print("-" * 60)
print(f"{'Eigenfaces (nearest projection)':<40}{acc_ef:>20.1%}")
print(f"{'Least squares':<40}{acc_ls:>20.1%}")
print(f"{'CNN':<40}{acc_cnn:>20.1%}")
print("=" * 60)

**Workflow recap:**
Real-world data (face photos) → **matrix representation** (Step 1) → **centering** (Step 2) → **rank** (Step 3) →
**eigenvectors & diagonalization / PCA** (Step 4) → **eigenfaces** (Step 5) → **projection onto a subspace** (Step 6) →
**face recognition** (Steps 7–8, including least squares) → comparison with a CNN (Step 9).